# Local GPU Multi-Sample Clustering Sweep

Runs clustering sweeps with **local GPU embedding engines** across multiple sample sizes.
Uses `LocalDockerTEIManager` to start/stop a TEI container per model automatically.

- **Datasets**: dbpedia, yahoo_answers
- **Sample sizes**: 100, 500, 1000
- **Embeddings**: 14 local TEI models (one container per model, auto-managed)
- **Summarizers**: None, gpt-4o-mini, gpt-4o, gpt-5-chat, claude-opus-4-6
- **Config**: k=2–20, 50 restarts, skip_pca=True, cosine, normalize
- **Total runs**: 420 (2 datasets × 3 sizes × 14 engines × 5 summarizers)

After sweeps complete, the final cell runs `ingest_sweep_to_db.py` to load results into the database.

In [ ]:
%pip install -q openai python-dotenv sqlalchemy psycopg2-binary nest_asyncio tqdm datasets docker

## Setup paths and environment

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "src" / "study_query_llm").exists() else cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

DATABASE_URL = os.environ.get("DATABASE_URL")
if not DATABASE_URL:
    raise ValueError("DATABASE_URL environment variable is required")

print("REPO_ROOT:", REPO_ROOT)
print("DATABASE_URL set:", bool(DATABASE_URL))

## Configuration

In [ ]:
SAMPLE_SIZES = [100, 500, 1000]
N_RESTARTS = 50
K_MIN, K_MAX = 2, 20
OUT_PREFIX = "local_gpu_multi"

EMBEDDING_ENGINES = [
    "Qwen/Qwen3-Embedding-0.6B",
    "Qwen/Qwen3-Embedding-4B",
    "Qwen/Qwen3-Embedding-8B",
    "Alibaba-NLP/gte-Qwen2-7B-instruct",
    "intfloat/multilingual-e5-large-instruct",
    "Alibaba-NLP/gte-Qwen2-1.5B-instruct",
    "Snowflake/snowflake-arctic-embed-l-v2.0",
    "WhereIsAI/UAE-Large-V1",
    "BAAI/bge-m3",
    "BAAI/bge-large-en-v1.5",
    "Alibaba-NLP/gte-large-en-v1.5",
    "nomic-ai/nomic-embed-text-v1.5",
    "nomic-ai/nomic-embed-text-v2-moe",
    "sentence-transformers/all-mpnet-base-v2",
]

SUMMARIZERS = [None, "gpt-4o-mini", "gpt-4o", "gpt-5-chat", "claude-opus-4-6"]

DATASET_CONFIGS = [
    {"name": "dbpedia", "label_max": 14},
    {"name": "yahoo_answers", "label_max": 10},
]

total_runs = len(DATASET_CONFIGS) * len(SAMPLE_SIZES) * len(EMBEDDING_ENGINES) * len(SUMMARIZERS)
print(f"Total runs: {total_runs} ({len(DATASET_CONFIGS)} datasets x {len(SAMPLE_SIZES)} sizes x {len(EMBEDDING_ENGINES)} engines x {len(SUMMARIZERS)} summarizers)")

## Imports and sweep config

In [ ]:
import asyncio
import nest_asyncio
import numpy as np
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

nest_asyncio.apply()

from study_query_llm.db.connection_v2 import DatabaseConnectionV2
from study_query_llm.db.raw_call_repository import RawCallRepository
from study_query_llm.services.embedding_service import EmbeddingService, EmbeddingRequest
from study_query_llm.services.paraphraser_factory import create_paraphraser_for_llm
from study_query_llm.algorithms import SweepConfig, run_sweep
from study_query_llm.experiments.sweep_io import save_single_sweep_result, get_output_dir
from study_query_llm.providers.managers.local_docker_tei import LocalDockerTEIManager
from study_query_llm.providers.managed_tei_embedding_provider import ManagedTEIEmbeddingProvider

def load_dbpedia_full(random_state=42):
    from datasets import load_dataset
    dataset = load_dataset("dbpedia_14", split="train")
    texts, labels = [], []
    for item in dataset:
        text, label = item.get("content", ""), item.get("label", -1)
        if text and 10 < len(text) <= 1000 and label >= 0:
            texts.append(text); labels.append(label)
    cats = ["Company", "EducationalInstitution", "Artist", "Athlete", "OfficeHolder", "MeanOfTransportation", "Building", "NaturalPlace", "Village", "SportsTeam", "Information", "Animal", "Plant", "Album"]
    return texts, np.array(labels), cats

def load_yahoo_answers_full(random_state=42):
    from datasets import load_dataset
    dataset = load_dataset("yahoo_answers_topics", split="train")
    texts, labels = [], []
    for item in dataset:
        text = f"{item.get('question_title','')}\n{item.get('question_content','')}\n{item.get('best_answer','')}".strip()
        label = item.get("topic", -1)
        if text and 10 < len(text) <= 1000 and label >= 0:
            texts.append(text); labels.append(label)
    cats = ["Society & Culture", "Science & Mathematics", "Health", "Education & Reference", "Computers & Internet", "Sports", "Business & Finance", "Entertainment & Music", "Family & Relationships", "Politics & Government"]
    return texts, np.array(labels), cats

LOADERS = {
    "dbpedia": load_dbpedia_full,
    "yahoo_answers": load_yahoo_answers_full,
}

OUTPUT_DIR = get_output_dir()

SWEEP_CONFIG = SweepConfig(
    skip_pca=True,
    k_min=K_MIN,
    k_max=K_MAX,
    max_iter=200,
    base_seed=0,
    n_restarts=N_RESTARTS,
    compute_stability=True,
    llm_interval=20,
    max_samples=10,
    distance_metric="cosine",
    normalize_vectors=True,
)

def _safe_name(s: str) -> str:
    return s.replace("-", "_").replace("/", "_")

print("[OK] Imports and config ready")

## Load datasets (full, once)

In [ ]:
loaded_full = {}
for cfg in DATASET_CONFIGS:
    name = cfg["name"]
    print(f"Loading {name}...")
    try:
        texts_all, labels_all, category_names = LOADERS[name]()
    except Exception as exc:
        print(f"  [ERROR] {exc}")
        continue
    loaded_full[name] = {
        "texts_all": texts_all,
        "labels_all": labels_all,
        "label_max": cfg["label_max"],
        "category_names": category_names,
    }
    print(f"  {len(texts_all)} texts, {len(set(labels_all))} unique labels")

print(f"\n[OK] Loaded {len(loaded_full)} datasets")

## Run sweeps

For each embedding engine, a Docker TEI container is started with that specific model,
embeddings are fetched, then sweeps run for all sample sizes / datasets / summarizers.
The container is stopped after each engine to free VRAM.

In [ ]:
db = DatabaseConnectionV2(DATABASE_URL, enable_pgvector=True)
db.init_db()

run_count = 0
total = total_runs

async def _fetch_embeddings(texts, engine, emb_provider):
    """Fetch embeddings using the managed TEI provider."""
    with db.session_scope() as session:
        repo = RawCallRepository(session)
        service = EmbeddingService(repository=repo, provider=emb_provider)
        requests = [
            EmbeddingRequest(
                text=t,
                deployment=engine,
                provider=emb_provider.get_provider_name(),
            )
            for t in texts
        ]
        responses = await service.get_embeddings_batch(requests, chunk_size=32)
        return np.asarray([r.vector for r in responses], dtype=np.float64)

for embedding_engine in EMBEDDING_ENGINES:
    engine_safe = _safe_name(embedding_engine)
    print("\n" + "=" * 80)
    print(f"ENGINE: {embedding_engine}")
    print("=" * 80)

    with LocalDockerTEIManager(model_id=embedding_engine, use_gpu=True) as manager:
        loop = asyncio.get_event_loop()

        async def _create_provider():
            return ManagedTEIEmbeddingProvider(manager)

        emb_provider = loop.run_until_complete(_create_provider())

        try:
            for entry_max in SAMPLE_SIZES:
                for dataset_name in [cfg["name"] for cfg in DATASET_CONFIGS]:
                    full = loaded_full.get(dataset_name)
                    if full is None:
                        run_count += len(SUMMARIZERS)
                        continue

                    texts_all = full["texts_all"]
                    labels_all = full["labels_all"]
                    label_max = full["label_max"]

                    unique_labels = sorted(set(labels_all))
                    lm = min(label_max, len(unique_labels))
                    mask = np.isin(labels_all, unique_labels[:lm])
                    idx = np.where(mask)[0]

                    if len(idx) > entry_max:
                        np.random.seed(42)
                        idx = np.random.choice(idx, size=entry_max, replace=False)

                    texts = [texts_all[i] for i in idx]
                    labels = labels_all[idx]

                    valid = [i for i, t in enumerate(texts) if 10 < len(t) <= 1000]
                    if len(valid) < len(texts):
                        texts = [texts[i] for i in valid]
                        labels = labels[valid]

                    gt = labels
                    print(f"\n  {dataset_name} n={len(texts)} (entry_max={entry_max})")

                    try:
                        embeddings = loop.run_until_complete(
                            _fetch_embeddings(texts, embedding_engine, emb_provider)
                        )
                    except Exception as exc:
                        print(f"    [ERROR] Embedding failed: {exc}")
                        run_count += len(SUMMARIZERS)
                        continue

                    def _embed_sync(txts):
                        return loop.run_until_complete(
                            _fetch_embeddings(txts, embedding_engine, emb_provider)
                        )

                    for summarizer_name in SUMMARIZERS:
                        run_count += 1
                        sum_safe = _safe_name(str(summarizer_name))
                        out_name = f"{OUT_PREFIX}_entry{entry_max}_{dataset_name}_{engine_safe}_{sum_safe}_"

                        if list(OUTPUT_DIR.glob(out_name + "[0-9]*.pkl")):
                            print(f"    [{run_count}/{total}] {summarizer_name} (SKIP \u2013 pkl exists)")
                            continue

                        print(f"    [{run_count}/{total}] {summarizer_name}")

                        paraphraser = None
                        if summarizer_name:
                            try:
                                paraphraser = create_paraphraser_for_llm(summarizer_name, db)
                            except Exception as exc:
                                print(f"      [WARN] Paraphraser failed: {exc}")

                        try:
                            with ThreadPoolExecutor() as executor:
                                result = loop.run_until_complete(
                                    loop.run_in_executor(
                                        executor,
                                        lambda: run_sweep(
                                            texts, embeddings, SWEEP_CONFIG,
                                            paraphraser=paraphraser,
                                            embedder=_embed_sync if paraphraser else None,
                                        ),
                                    )
                                )
                        except Exception as exc:
                            print(f"      [ERROR] Sweep failed: {exc}")
                            continue

                        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                        out_path = OUTPUT_DIR / f"{out_name}{ts}.pkl"
                        metadata = {
                            "embedding_engine": embedding_engine,
                            "embedding_provider": "local_docker_tei",
                            "summarizer": str(summarizer_name),
                            "n_restarts": N_RESTARTS,
                            "k_min": K_MIN,
                            "k_max": K_MAX,
                            "entry_max": entry_max,
                            "dataset_name": dataset_name,
                            "label_max": label_max,
                            "n_texts": len(texts),
                            "distance_metric": "cosine",
                            "normalize_vectors": True,
                            "skip_pca": True,
                        }

                        save_single_sweep_result(
                            result, str(out_path),
                            ground_truth_labels=gt,
                            dataset_name=dataset_name,
                            metadata=metadata,
                        )
                        print(f"      Saved: {out_path.name}")

        finally:
            loop.run_until_complete(emb_provider.close())

    print(f"  [Container stopped for {embedding_engine}]")

print(f"\n[DONE] {run_count}/{total} runs processed")

## Ingest results to database

Runs `scripts/ingest_sweep_to_db.py` to load all pkl results into the database and create a clustering sweep group.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "scripts/ingest_sweep_to_db.py", "--data-dir", str(REPO_ROOT / "experimental_results")],
    cwd=str(REPO_ROOT),
    capture_output=False,
)
print(f"\nIngest exited with code {result.returncode}")